# Lab 1 — The Model Zoo: What IS a Prediction?

---

> **Lessons covered**: 01 Narrow vs General AI · 02 Predictive Systems · 03 Discriminative Models
> **Metric introduced**: Accuracy (and why it lies)

---

## Learning Goal

We **start with a prediction from an LLM**, then compare it to classical models. Five completely different algorithms — one task, one output format.
By the end of this lab you will understand:

1. What an LLM returns for a simple approve/deny question (and why that matters)
2. Why loan approval is a **narrow AI** task: fixed input space, fixed output space (`approve` / `deny`)
3. How four different **discriminative models** each find a different boundary between approved and denied
4. Why **accuracy** is the intuitive first metric — and exactly why it is dangerous to stop there

---

**Instructions**: Visualisations are provided or have minimal TODOs. The **exercises** are about **using** the information: read the plots, answer the questions, and run the comparison experiments (e.g. drop a feature and see how the result changes). Run all cells top-to-bottom. If you get stuck, check `solution.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from openai import OpenAI

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print('All imports OK')

---
## Section 1 — Load & Explore the Dataset

Before building any model we need to understand the data.
At this point we only use **four features** — the minimum viable prediction set.

| Feature | Type | Meaning |
|---|---|---|
| `credit_score` | int 300–850 | Borrower credit history score |
| `annual_income` | float (€) | Verified yearly income |
| `loan_amount` | float (€) | Amount requested |
| `num_defaults` | int 0–5 | Past payment defaults |
| **`approved`** | **0 / 1** | **Target label — what we predict** |

> **Why narrow AI?**  
> The input is a fixed set of structured fields. The output is always 0 or 1.  
> Nothing else. The model cannot write text, reason about context, or handle  
> any task outside this scope. That is the definition of **narrow AI**.

In [ ]:
DATA_PATH = Path('../data/loan_applications.csv')

# TODO: Load the CSV into a DataFrame called df
# Hint: pd.read_csv(...)
df = None  # replace None with your code

# Lab 1 uses only these four features
FEATURES = ['credit_score', 'annual_income', 'loan_amount', 'num_defaults']
TARGET   = 'approved'

print(f'Dataset: {df.shape[0]} applicants, {df.shape[1]} total columns')

# TODO: Show descriptive statistics for FEATURES + TARGET
# Hint: df[FEATURES + [TARGET]].describe().round(2)


In [ ]:
# Exploration figure (provided) — use it to see feature distributions and class balance
fig = plt.figure(figsize=(18, 8))
gs  = fig.add_gridspec(2, 4, hspace=0.4, wspace=0.4)
colors_feat = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']
for col, feat, c in zip(range(4), FEATURES, colors_feat):
    ax = fig.add_subplot(gs[0, col])
    ax.hist(df[feat], bins=30, color=c, alpha=0.8, edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
ax_bal = fig.add_subplot(gs[1, :2])
balance = df[TARGET].value_counts().rename({0: 'Denied (0)', 1: 'Approved (1)'})
bars = ax_bal.bar(balance.index, balance.values, color=['#e74c3c', '#2ecc71'], edgecolor='white', width=0.4)
ax_bal.set_ylabel('Number of applicants')
ax_bal.set_title('Class Balance — 65% Approved · 35% Denied', fontsize=11)
for bar, v in zip(bars, balance.values):
    ax_bal.text(bar.get_x() + bar.get_width()/2, v + 8, str(v), ha='center', fontweight='bold')
ax_sc = fig.add_subplot(gs[1, 2:])
for label, grp in df.groupby(TARGET):
    ax_sc.scatter(grp['credit_score'], grp['loan_amount']/1000, alpha=0.3, s=15,
                  color='#2ecc71' if label == 1 else '#e74c3c',
                  label='Approved' if label == 1 else 'Denied')
ax_sc.set_xlabel('Credit Score')
ax_sc.set_ylabel('Loan Amount (k€)')
ax_sc.set_title('Credit Score vs Loan Amount by Outcome', fontsize=11)
ax_sc.legend(markerscale=2)
fig.suptitle('Exploratory Data Analysis — Loan Applications', fontsize=14, y=1.01)
plt.show()
print(f'Approval rate: {df[TARGET].mean():.1%}   ← remember this number')


In [ ]:
# Train / test split (already done for you — understand each parameter)
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,      # 20% held out for evaluation
    random_state=42,     # reproducibility
    stratify=y           # preserve 65/35 split in both sets
)

# TODO: Fit a StandardScaler on X_train and transform both X_train and X_test
# Name the results X_train_sc and X_test_sc
# Hint: scaler = StandardScaler()  then  fit_transform / transform

scaler = StandardScaler()
X_train_sc = None  # replace with: scaler.fit_transform(X_train)
X_test_sc  = None  # replace with: scaler.transform(X_test)

print(f'Train: {len(X_train)} samples | Test: {len(X_test)} samples')
print(f'Test approval rate: {y_test.mean():.1%}')

---
## Section 0 — A first prediction with an LLM

Before we train classical models, let's see what an **LLM** would say. We take one applicant from the test set and ask the model: approve or deny?

**How to use this**: The LLM returns **text**, not 0/1. We have to parse it. Lab 2 will compare the LLM to XGBoost on the full test set — here we just get a taste.

In [ ]:
# One applicant from the test set (provided)
applicant = df[FEATURES].loc[X_test.index[0]].to_dict()
true_label = int(y_test.iloc[0])
print('Applicant:')
for k, v in applicant.items():
    print(f'  {k}: {v}')
print(f'  True label: {true_label} (1 = Approved, 0 = Denied)')

# Build prompt and call LLM (provided — run to see the result)
prompt = (
    "You are a loan officer. Based only on these numbers, should this loan be approved?\n"
    + "\n".join(f"{k}: {v}" for k, v in applicant.items())
    + "\n\nAnswer with only: Approve or Deny."
)
try:
    client = OpenAI()
    resp = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=10,
    )
    raw_text = resp.choices[0].message.content.strip()
    llm_approve = 1 if raw_text.lower().startswith('approv') else 0
    print(f'\nLLM raw response: "{raw_text}"')
    print(f'Parsed as: {llm_approve} (1=Approved, 0=Denied)')
    print(f'Match: {"Yes" if llm_approve == true_label else "No"}')
except Exception as e:
    print(f'\n(API not available: {e}. Set OPENAI_API_KEY to run the LLM taster.)')
    raw_text = None
    llm_approve = None

# EXERCISE: What did the LLM predict? What was the true label?
# In one sentence: what does this show about using an LLM for this task?

---
## Section 2 — Model 1: Logistic Regression

**How it works**  
Learns a *weighted sum* of features:
```
approval_score = w₁ × credit_score + w₂ × income + w₃ × loan_amount + w₄ × defaults + bias
```
If `approval_score > threshold` → Approve. The decision boundary is a hyperplane.

**Why start here**
- Fastest to train
- Coefficients give direct explanations
- Required by many banking regulators as an interpretable baseline

**All four models in this lab are *discriminative***  
They learn `P(approved | features)` — a direct mapping from input to label.

**How to use the coefficient plot**  
The bar chart shows how each feature pushes the decision: **green** = raises approval probability, **red** = lowers it. Use it to see which feature the model relies on most (largest magnitude) and in which direction. You can use this to explain a denial to an applicant or to decide which feature to drop in an experiment.

In [2]:
# TODO: Train a LogisticRegression model (random_state=42, max_iter=1000)
# on X_train_sc and y_train. Predict on X_test_sc and compute accuracy.
# Store results as: lr, lr_preds, lr_acc

# YOUR CODE HERE

print(f'Logistic Regression — Test Accuracy: {lr_acc:.3f}')

# Coefficient plot (provided) — use it to answer the exercise below
coef_df = pd.DataFrame({'feature': FEATURES, 'weight': lr.coef_[0]})
coef_df = coef_df.sort_values('weight', key=abs, ascending=True)
fig, ax = plt.subplots(figsize=(8, 3.5))
bar_c = ['#2ecc71' if w > 0 else '#e74c3c' for w in coef_df['weight']]
ax.barh(coef_df['feature'], coef_df['weight'], color=bar_c, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient weight')
ax.set_title('Logistic Regression: What Does the Model Think Matters?\nGreen = raises approval  |  Red = lowers it')
plt.tight_layout()
plt.show()

# EXERCISE (use the plot): (1) Which feature has the largest positive impact? The smallest (or most negative)?
# (2) In one sentence, what would you tell an applicant who was denied?


NameError: name 'lr_acc' is not defined

In [ ]:
# EXERCISE — Use the coefficient plot: which feature has the SMALLEST |coefficient|?
# Train LR again WITHOUT that feature; report test accuracy. Then drop the feature with the LARGEST
# |coefficient| and report again. Which removal hurt accuracy more? (Your answer should match the plot.)
def idx_smallest_coef():
    c = np.abs(lr.coef_[0])
    return np.argmin(c)
def idx_largest_coef():
    c = np.abs(lr.coef_[0])
    return np.argmax(c)

drop_least = [f for i, f in enumerate(FEATURES) if i != idx_smallest_coef()]
drop_most  = [f for i, f in enumerate(FEATURES) if i != idx_largest_coef()]

X_train_least = X_train[drop_least]
X_test_least  = X_test[drop_least]
scaler_least = StandardScaler()
X_train_least_sc = scaler_least.fit_transform(X_train_least)
X_test_least_sc  = scaler_least.transform(X_test_least)
lr_least = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_least_sc, y_train)
acc_least = accuracy_score(y_test, lr_least.predict(X_test_least_sc))

X_train_most = X_train[drop_most]
X_test_most  = X_test[drop_most]
scaler_most = StandardScaler()
X_train_most_sc = scaler_most.fit_transform(X_train_most)
X_test_most_sc  = scaler_most.transform(X_test_most)
lr_most = LogisticRegression(random_state=42, max_iter=1000).fit(X_train_most_sc, y_train)
acc_most = accuracy_score(y_test, lr_most.predict(X_test_most_sc))

print(f'Full model (4 features):     {lr_acc:.3f}')
print(f'Without least-important:     {acc_least:.3f}')
print(f'Without most-important:      {acc_most:.3f}')
print('Which removal hurt more? Your answer:')

---
## Section 3 — Model 2: Decision Tree

Learns explicit if-then-else rules. The most explainable model in the zoo.

In regulated lending, the EU AI Act and Consumer Credit Directive require explainable automated decisions.  
A decision tree is the only model here that generates a natural explanation by itself.

**How to use the tree plot**  
Each path from root to leaf is a rule. Use the tree to **explain a single decision**: pick one test applicant, follow the path (left/right at each split according to their feature values), and see which leaf they land in and whether the model approves or denies. Then check with `dt.predict()` — your path should match.

In [ ]:
# TODO: Train a DecisionTreeClassifier (max_depth=4, random_state=42)
# Note: trees do NOT need scaled features — train on X_train (not X_train_sc)
# Predict on X_test and compute accuracy. Store as: dt, dt_preds, dt_acc

# YOUR CODE HERE

print(f'Decision Tree (max_depth=4) — Test Accuracy: {dt_acc:.3f}')

# Tree visualisation (provided)
fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(dt, feature_names=FEATURES, class_names=['Denied', 'Approved'],
          filled=True, rounded=True, fontsize=9, impurity=False, ax=ax)
plt.tight_layout()
plt.show()

# EXERCISE (use the tree): Pick test applicant row 0. By following the tree from the root (compare
# their feature values to each split), which leaf do they land in? What does the model predict?
# Check: does dt.predict(X_test.iloc[:1]) match your answer?
print('Test applicant 0:', X_test.iloc[0].to_dict())
print('Model prediction:', dt.predict(X_test.iloc[:1])[0], '(1=Approved, 0=Denied)')
print('Your turn: trace this applicant through the tree above and confirm the prediction.')


---
## Section 4 — Model 3: K-Nearest Neighbours (KNN)

Stores the training set and votes based on similar historical applicants.

**⚠ Common misconception — corrected**

| Claim | True? |
|---|---|
| KNN is unsupervised | ❌ **False**. KNN needs labels to vote. It IS supervised. |
| KNN trains slowly | ❌ **False**. Zero training time — it just memorises. |
| KNN predicts slowly | ✅ True. Must scan all N training points per query. |

**How to use the K vs accuracy plot**  
The curve shows how many neighbours (K) gives the best accuracy. Too low K → noisy (overfitting); too high K → too smooth (underfitting). Use the plot to choose K and to see why accuracy drops at K=1 and at high K.

In [ ]:
# TODO: Sweep K from 1 to 30, compute test accuracy for each K
# Store results in k_values and k_accuracies. Use X_train_sc and X_test_sc (KNN is sensitive to scale)
k_values     = list(range(1, 31))
k_accuracies = []
for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_sc, y_train)
    k_accuracies.append(accuracy_score(y_test, m.predict(X_test_sc)))

best_k   = k_values[int(np.argmax(k_accuracies))]
best_acc = max(k_accuracies)
print(f'Best K = {best_k}  |  Accuracy = {best_acc:.3f}')

# K vs accuracy plot (provided)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_values, k_accuracies, marker='o', markersize=4, color='#9b59b6', linewidth=1.5)
ax.axvline(best_k, color='#e74c3c', linestyle='--', linewidth=1.5, label=f'Best K = {best_k}')
ax.set_xlabel('K (number of neighbours)')
ax.set_ylabel('Test Accuracy')
ax.set_title('KNN: Accuracy vs K')
ax.legend()
plt.tight_layout()
plt.show()

# EXERCISE (use the plot): What is the best K and (roughly) the worst K in 1–30?
# In one sentence: why does accuracy get worse at K=1 and at high K?


In [ ]:
# TODO: Train the final KNN with best_k, predict on X_test_sc, compute accuracy
# Store as: knn, knn_preds, knn_acc

# YOUR CODE HERE

print(f'KNN (k={best_k}) — Test Accuracy: {knn_acc:.3f}')

# Already done for you — inspect the nearest neighbours for 1 test applicant
sample_vec = X_test_sc[:1]
distances, indices = knn.kneighbors(sample_vec, n_neighbors=best_k)

neighbour_df = X_train.iloc[indices[0]].copy().reset_index(drop=True)
neighbour_df['outcome']  = ['✅ Approved' if v else '❌ Denied' for v in y_train.iloc[indices[0]].values]
neighbour_df['distance'] = distances[0].round(3)

print('\nSample applicant:')
print(X_test.iloc[0].to_frame().T.to_string(index=False))
print(f'\nNearest {best_k} historical neighbours:')
print(neighbour_df.to_string())

---
## Section 5 — Model 4: XGBoost

Builds trees **in sequence** — each tree corrects the errors of all previous trees.  
This is **gradient boosting**: fitting to the residual errors, guided by the loss gradient.

For structured tabular data, XGBoost is the performance ceiling.  
The LLM in Lab 2 will not beat this. **That is the point.**

**How to use the feature importance plot**  
It shows which inputs drive the model most. Use it to see what the model "cares about"; dropping the **least** important feature often barely changes accuracy; dropping the **most** important usually hurts a lot. The exercise below asks you to check this with a drop-feature experiment.

In [ ]:
# TODO: Train an XGBClassifier with:
#   n_estimators=100, learning_rate=0.1, max_depth=4,
#   random_state=42, eval_metric='logloss', verbosity=0
# Fit on X_train (no scaling needed for trees), with eval_set=[(X_test, y_test)]
# Store as: xgb, xgb_preds, xgb_acc

# YOUR CODE HERE

print(f'XGBoost — Test Accuracy: {xgb_acc:.3f}')

# 2-panel figure: training curve + feature importance (provided)
evals = xgb.evals_result()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(evals['validation_0']['logloss'], color='#e74c3c', linewidth=1.5)
ax1.set_xlabel('Round')
ax1.set_ylabel('Log loss')
ax1.set_title('XGBoost: Training curve')
importances = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values(ascending=True)
ax2.barh(importances.index, importances.values, color='#27ae60', edgecolor='white')
ax2.set_xlabel('Feature Importance (gain)')
ax2.set_title('XGBoost: Feature Importance — What Drives the Decision?')
plt.tight_layout()
plt.show()

# EXERCISE (use the importance plot): Which feature is MOST important? Which is LEAST?
# Below: train XGBoost again without the least important feature, then without the most important.
# Compare test accuracies. Which drop hurt more? (Should match the plot.)


In [ ]:
# Drop-feature experiment (use the importance plot to interpret)
imp = pd.Series(xgb.feature_importances_, index=FEATURES)
least_important = imp.idxmin()
most_important  = imp.idxmax()
feat_least = [f for f in FEATURES if f != least_important]
feat_most  = [f for f in FEATURES if f != most_important]

xgb_least = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss', verbosity=0)
xgb_least.fit(X_train[feat_least], y_train, eval_set=[(X_test[feat_least], y_test)], verbose=False)
acc_least_xgb = accuracy_score(y_test, xgb_least.predict(X_test[feat_least]))

xgb_most = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42, eval_metric='logloss', verbosity=0)
xgb_most.fit(X_train[feat_most], y_train, eval_set=[(X_test[feat_most], y_test)], verbose=False)
acc_most_xgb = accuracy_score(y_test, xgb_most.predict(X_test[feat_most]))

print(f'Full XGBoost:              {xgb_acc:.3f}')
print(f'Without "{least_important}": {acc_least_xgb:.3f}')
print(f'Without "{most_important}":  {acc_most_xgb:.3f}')
print('Which removal hurt more? Your answer (should match the importance plot):')

---
## Section 6 — The Grand Comparison

Build a summary table and visualise all four models side-by-side.

**How to use this plot**  
Use the bar chart to compare models at a glance. Section 7 will add the dummy model — then you'll see why accuracy alone is misleading.

**Think first**: Which model will win? By how much?  
**EXERCISE**: From the plot, which model has the highest accuracy? If you had to deploy one at a regulated bank tomorrow, which would you choose and why? (One sentence.)

In [ ]:
# Summary table and comparison plot (provided) — use the plot to answer the exercise in the section above
results_df = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_acc,  'Explainable': 'Yes', 'Trains fast': 'Yes'},
    {'Model': f'KNN (k={best_k})',    'Accuracy': knn_acc, 'Explainable': 'Partial', 'Trains fast': 'Yes (lazy)'},
    {'Model': 'Decision Tree',        'Accuracy': dt_acc,  'Explainable': 'Best', 'Trains fast': 'Yes'},
    {'Model': 'XGBoost',             'Accuracy': xgb_acc, 'Explainable': 'No',   'Trains fast': 'Yes'},
]).sort_values('Accuracy', ascending=True).reset_index(drop=True)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4.5))
colors = ['#9b59b6', '#3498db', '#f39c12', '#27ae60']
bars = ax.barh(results_df['Model'], results_df['Accuracy'], color=colors, edgecolor='white', height=0.5)
ax.set_xlabel('Test Accuracy')
ax.set_title('Model Comparison — Test Set Accuracy')
ax.set_xlim(0.55, 1.00)
for bar, val in zip(bars, results_df['Accuracy']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center', fontsize=11)
plt.tight_layout()
plt.show()


---
## Section 7 — The Accuracy Trap ⚠️

**The most important section in Lab 1.**

Recall from Section 1: **65% of applicants are approved**.

Here is a bold claim: we can build a model that requires zero training, zero intelligence — and it will score **65% accuracy**.

**How to use the confusion matrices**  
They show *where* the model is wrong (true vs predicted). Use them to see that the dummy **never** predicts Denied, while XGBoost does — accuracy hides that completely.

> **Challenge**: Before running the next cell, predict:  
> Will the dummy model beat any of our four trained models?

In [ ]:
# TODO: Create a DummyClassifier(strategy='most_frequent'), train it, predict on X_test, compute accuracy.
# Store as: dummy, dummy_preds, dummy_acc

# YOUR CODE HERE

print(f'DummyClassifier (always says "Approved") — Accuracy: {dummy_acc:.3f}')
print()

# Add dummy to comparison and re-plot (provided)
all_results = pd.DataFrame([
    {'Model': 'Logistic Regression', 'Accuracy': lr_acc},
    {'Model': f'KNN (k={best_k})', 'Accuracy': knn_acc},
    {'Model': 'Decision Tree', 'Accuracy': dt_acc},
    {'Model': 'XGBoost', 'Accuracy': xgb_acc},
    {'Model': 'Dummy (always Approve)', 'Accuracy': dummy_acc},
]).sort_values('Accuracy', ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#9b59b6', '#3498db', '#f39c12', '#27ae60', '#e74c3c']
bars = ax.barh(all_results['Model'], all_results['Accuracy'], color=bar_colors, edgecolor='white', height=0.5)
ax.axvline(dummy_acc, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7, label=f'Dummy = {dummy_acc:.2f}')
ax.set_xlabel('Test Accuracy')
ax.set_title('Model Comparison — Including the Dummy')
ax.set_xlim(0.55, 1.00)
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices: XGBoost vs Dummy (provided) — use them to answer the exercise below
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, preds, name in [(axes[0], xgb_preds, 'XGBoost'), (axes[1], dummy_preds, 'Dummy (always Approve)')]:
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Denied', 'Approved'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

print('EXERCISE (use the two panels): How many "Denied" predictions does the dummy make?')
print('How many does XGBoost make? Why can they have similar accuracy?')
print()
print('Key insight: Accuracy treats every prediction equally.')
print('The dummy NEVER predicts "Denied" — accuracy hides this completely.')
print()
print('Question for Lab 2: The bank loses more money on an approved defaulter')
print('than it gains from an approved good customer. Does accuracy capture this? (No.)')
print('What SHOULD we measure? → Lab 2 will answer this.')